In [ ]:
# In[1]:

import pandas as pd

# Load files
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)
log['timestamp'] = pd.to_datetime(log['timestamp'], unit='s', utc=True)
error_logs['timestamp'] = pd.to_datetime(error_logs['timestamp'], unit='s', utc=True)

# 1) metric.csv: aggregate by (cmdb_id, kpi_name) and compute count and P95
metrics_thresholds = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .agg(count='count', p95=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# 2) trace.csv: aggregate by (cmdb_id, trace_name) and compute count and P95
traces_thresholds = (
    trace
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .agg(count='count', p95=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# 3) log.csv: aggregate by (cmdb_id, log_name) and compute count and P95
logs_thresholds = (
    log
    .groupby(['cmdb_id', 'log_name'], observed=True)['value']
    .agg(count='count', p95=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# 4) distinct cmdb_id values across all files
all_ids = pd.concat([
    metric['cmdb_id'],
    trace['cmdb_id'],
    log['cmdb_id'],
    error_logs['cmdb_id']
], ignore_index=True).dropna().astype(str)
distinct_cmdb_ids = pd.DataFrame(sorted(all_ids.unique()), columns=['cmdb_id'])

# 5) top 5 rows from error_logs.csv sorted by timestamp asc
error_logs_top5 = error_logs.sort_values('timestamp', ascending=True)[['timestamp', 'cmdb_id', 'message']].head(5)

# Return compact results
metrics_thresholds, traces_thresholds, logs_thresholds, distinct_cmdb_ids, error_logs_top5

```
Out[1]:
```
```python
# Summarize the previously computed results in plain English.
ids_list = ", ".join(distinct_cmdb_ids['cmdb_id'].tolist())

summary = (
    "Summary of telemetry thresholds and observations:\n"
    "- Metric thresholds (top by data points): adservice, cartservice, checkoutservice.\n"
    "  * adservice: very high P95 memory (~1.10e8) and very high disk I/O (~1.05e5); socket and workload also notable.\n"
    "  * cartservice: high P95 memory (~4.36e7) and elevated CPU/workload values.\n"
    "  * checkoutservice: noticeable latency P95 (~2.07 s) and high socket count (9); memory P95 is lower than ad/cart but nontrivial.\n"
    "- Trace thresholds: checkoutservice dominates trace metrics. Notable p95 durations include ~1.85 s for traces from frontend to checkoutservice and row_count p95 up to ~274, indicating backend latency/volume on checkoutservice paths.\n"
    "- Log thresholds: frontend and currencyservice have the largest log row_counts (frontend ~2804, currencyservice ~2527); log.error_count P95 is 0 for these services.\n"
    f"- Distinct services observed across files: {ids_list}.\n"
    "- Error logs: no error_log rows returned (empty result).\n\n"
    "Conclusion / next steps: checkoutservice is the top candidate for latency-related root cause (high trace latency p95). "
    "Adservice and cartservice show very high memory and disk I/O P95s and should be investigated for resource issues. "
    "Check logs and traces around checkoutservice for slow calls and inspect resource usage on adservice/cartservice."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id    kpi_name  count           p95
0         adservice         cpu     25  1.089928e+00
1         adservice      diskio     25  1.048093e+05
2         adservice  latency-50     25  3.017253e-03
3         adservice  latency-90     25  4.640888e-03
4         adservice         mem     25  1.098053e+08
5         adservice      socket     25  4.000000e+00
6         adservice    workload     25  1.550042e+01
7       cartservice         cpu     25  2.799373e+00
8       cartservice  latency-50     25  3.122909e-03
9       cartservice  latency-90     25  4.821235e-03
10      cartservice         mem     25  4.355621e+07
11      cartservice      socket     25  5.000000e+00
12      cartservice    workload     25  2.658646e+01
13  checkoutservice         cpu     25  4.515477e-01
14  checkoutservice      diskio     25  0.000000e+00
15  checkoutservice  latency-50     25  3.957373e-01
16  checkoutservice  latency-90     25  2.070605e+00
17  checkoutservice         mem     25  1.215600e+07
18  checkoutservice      socket     25  9.000000e+00
19  checkoutservice    workload     25  1.154737e+00,             cmdb_id                                trace_name  count         p95
0   checkoutservice  trace.from_checkoutservice.duration_mean     25    0.072269
1   checkoutservice   trace.from_checkoutservice.duration_p95     25    0.590802
2   checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25  274.000000
4   checkoutservice  trace.from_frontendservice.duration_mean     25    0.701483
5   checkoutservice   trace.from_frontendservice.duration_p95     25    1.852933
6   checkoutservice     trace.from_frontendservice.error_rate     25    0.000000
7   checkoutservice      trace.from_frontendservice.row_count     25   32.800000
8   checkoutservice             trace.from_root.duration_mean     25    0.014515
9   checkoutservice              trace.from_root.duration_p95     25    0.069447
10  checkoutservice                trace.from_root.error_rate     25    0.000000
11  checkoutservice                 trace.from_root.row_count     25  146.600000
12  checkoutservice    trace.to_checkoutservice.duration_mean     25    0.072269
13  checkoutservice     trace.to_checkoutservice.duration_p95     25    0.590802
14  checkoutservice       trace.to_checkoutservice.error_rate     25    0.000000
15  checkoutservice        trace.to_checkoutservice.row_count     25  274.000000
16  checkoutservice    trace.to_currencyservice.duration_mean     25    0.002045
17  checkoutservice     trace.to_currencyservice.duration_p95     25    0.000275
18  checkoutservice       trace.to_currencyservice.error_rate     25    0.000000
19  checkoutservice        trace.to_currencyservice.row_count     25  122.600000,                   cmdb_id         log_name  count     p95
0               adservice  log.error_count     25     0.0
1               adservice    log.row_count     25   464.4
2             cartservice  log.error_count     25     0.0
3             cartservice    log.row_count     25   808.2
4         checkoutservice  log.error_count     25     0.0
5         checkoutservice    log.row_count     25    98.6
6         currencyservice  log.error_count     25     0.0
7         currencyservice    log.row_count     25  2527.2
8            emailservice  log.error_count     25     0.0
9            emailservice    log.row_count     25    33.0
10               frontend  log.error_count     25     0.0
11               frontend    log.row_count     25  2804.0
12         paymentservice  log.error_count     25     0.0
13         paymentservice    log.row_count     25    65.6
14  recommendationservice  log.error_count     25     0.0
15  recommendationservice    log.row_count     25   593.2
19        shippingservice    log.row_count     25   511.2
18        shippingservice  log.error_count     25     0.0
17                  redis    log.row_count      5     5.0
16                  redis  log.error_count      5     0.0,                   cmdb_id
0               adservice
1             cartservice
2         checkoutservice
3         currencyservice
4            emailservice
5                frontend
6       frontend-external
7         frontendservice
8          paymentservice
9   productcatalogservice
10  recommendationservice
11                  redis
12                   root
13        shippingservice, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Load metric.csv and ensure timestamp is parsed as UTC datetime
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full series
global_p95 = (
    metric
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) Merge global_p95 back into the metric series
metric = metric.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Filter metric to the incident window (UTC, inclusive)
window_start = pd.Timestamp("2024-01-18 16:58:05", tz='UTC')
window_end   = pd.Timestamp("2024-01-18 17:28:05", tz='UTC')
df_window = metric[(metric['timestamp'] >= window_start) & (metric['timestamp'] <= window_end)].copy()

# 4) For each (cmdb_id, kpi_name) compute required summary fields
def summarize_group(g):
    count_window = len(g)
    exceed_mask = g['value'] > g['global_p95'].iloc[0]
    exceed_count = int(exceed_mask.sum())
    if exceed_count > 0:
        earliest_exceed_timestamp = g.loc[exceed_mask, 'timestamp'].min()
    else:
        earliest_exceed_timestamp = pd.NaT
    # max value and its timestamp (if multiple, pick the first occurrence)
    idxmax = g['value'].idxmax()
    max_value_in_window = g.loc[idxmax, 'value']
    max_value_timestamp = g.loc[idxmax, 'timestamp']
    return pd.Series({
        'cmdb_id': g['cmdb_id'].iloc[0],
        'kpi_name': g['kpi_name'].iloc[0],
        'global_p95': g['global_p95'].iloc[0],
        'count_window': count_window,
        'exceed_count': exceed_count,
        'earliest_exceed_timestamp': earliest_exceed_timestamp,
        'max_value_in_window': max_value_in_window,
        'max_value_timestamp': max_value_timestamp
    })

if not df_window.empty:
    metric_window_summary = (
        df_window
        .groupby(['cmdb_id', 'kpi_name'], observed=True, sort=False)
        .apply(summarize_group)
        .reset_index(drop=True)
    )
    # Keep only rows where count_window > 0 (should be all), sort and limit to top 20
    metric_window_summary = (
        metric_window_summary[metric_window_summary['count_window'] > 0]
        .sort_values(['exceed_count', 'max_value_in_window'], ascending=[False, False])
        .head(20)
        .reset_index(drop=True)
    )
else:
    # Empty result DataFrame with expected columns
    metric_window_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','count_window','exceed_count',
        'earliest_exceed_timestamp','max_value_in_window','max_value_timestamp'
    ])

# Display the compact summary (top results)
metric_window_summary

```
Out[2]:
```
Summary (incident window 2024-01-18 16:58:05 → 2024-01-18 17:28:05 UTC):

- Many services had values that exceeded their global P95 during the incident window (each reported 2 exceedances in the window). The dominant affected KPI is memory (mem) across multiple services:
  - mem exceeded global P95 for: emailservice, adservice, currencyservice, recommendationservice, cartservice, paymentservice, productcatalogservice, frontend, checkoutservice, shippingservice, redis.
  - Notable memory peaks (max value and time):
    - emailservice mem -> 2.272152e+08 at 2024-01-18 17:24:00 UTC
    - adservice mem -> 1.098075e+08 at 2024-01-18 17:24:00 UTC
    - currencyservice mem -> 4.678622e+07 at 2024-01-18 17:04:00 UTC
    - productcatalogservice mem -> 2.509339e+07 at 2024-01-18 17:23:00 UTC
    - redis mem -> 3.073980e+06 at 2024-01-18 17:10:00 UTC

- Other notable exceedances:
  - adservice diskio spiked above global P95, max 150,876.9 at 2024-01-18 17:04:00 UTC.
  - redis diskio also exceeded, max 11,914.94 at 2024-01-18 17:08:00 UTC.
  - Workload (request/row_count-like metric) spikes around 17:05 for productcatalogservice, currencyservice, cartservice, recommendationservice, adservice.
  - CPU exceedances observed for currencyservice (max ~32.57 at 17:01) and emailservice (max ~19.76 at 17:15).

- Timing pattern:
  - Many earliest exceedances occur around 17:01–17:05 UTC (workload/cpu/diskio) and a second wave around 17:21–17:25 UTC (memory peaks, e.g., emailservice/adservice at 17:24).

Conclusion / recommended next steps:
- Memory pressure is widespread across services during the window and is the primary signal to investigate.
- Investigate adservice disk I/O spike at ~17:04 and workload spikes around ~17:05 (these may have triggered downstream memory/latency effects).
- Correlate these memory and I/O spikes with logs/traces around the noted timestamps (especially 17:04–17:05 and 17:21–17:25 UTC) to find root cause (e.g., sudden traffic surge, background job, memory leak, or I/O-heavy operation).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_name    global_p95  count_window  exceed_count earliest_exceed_timestamp  max_value_in_window       max_value_timestamp
0            emailservice       mem  2.226657e+08            25             2 2024-01-18 17:21:00+00:00         2.272152e+08 2024-01-18 17:24:00+00:00
1               adservice       mem  1.098053e+08            25             2 2024-01-18 17:24:00+00:00         1.098075e+08 2024-01-18 17:24:00+00:00
2         currencyservice       mem  4.665465e+07            25             2 2024-01-18 17:02:00+00:00         4.678622e+07 2024-01-18 17:04:00+00:00
3   recommendationservice       mem  4.542326e+07            25             2 2024-01-18 17:13:00+00:00         4.542601e+07 2024-01-18 17:13:00+00:00
4             cartservice       mem  4.355621e+07            25             2 2024-01-18 17:17:00+00:00         4.356321e+07 2024-01-18 17:20:00+00:00
5          paymentservice       mem  3.986369e+07            25             2 2024-01-18 17:24:00+00:00         3.987456e+07 2024-01-18 17:25:00+00:00
6   productcatalogservice       mem  2.486399e+07            25             2 2024-01-18 17:14:00+00:00         2.509339e+07 2024-01-18 17:23:00+00:00
7                frontend       mem  1.626046e+07            25             2 2024-01-18 17:22:00+00:00         1.637212e+07 2024-01-18 17:22:00+00:00
8         checkoutservice       mem  1.215600e+07            25             2 2024-01-18 17:01:00+00:00         1.215730e+07 2024-01-18 17:01:00+00:00
9         shippingservice       mem  9.108685e+06            25             2 2024-01-18 17:09:00+00:00         9.135787e+06 2024-01-18 17:09:00+00:00
10                  redis       mem  3.041867e+06            25             2 2024-01-18 17:05:00+00:00         3.073980e+06 2024-01-18 17:10:00+00:00
11              adservice    diskio  1.048093e+05            25             2 2024-01-18 17:04:00+00:00         1.508769e+05 2024-01-18 17:04:00+00:00
12                  redis    diskio  1.014632e+04            25             2 2024-01-18 17:08:00+00:00         1.191494e+04 2024-01-18 17:08:00+00:00
13  productcatalogservice  workload  1.475048e+02            25             2 2024-01-18 17:05:00+00:00         1.542056e+02 2024-01-18 17:05:00+00:00
14        currencyservice  workload  8.564420e+01            25             2 2024-01-18 17:05:00+00:00         8.663798e+01 2024-01-18 17:05:00+00:00
15        currencyservice       cpu  3.232162e+01            25             2 2024-01-18 17:01:00+00:00         3.257075e+01 2024-01-18 17:01:00+00:00
16            cartservice  workload  2.658646e+01            25             2 2024-01-18 17:05:00+00:00         2.739992e+01 2024-01-18 17:05:00+00:00
17  recommendationservice  workload  1.929510e+01            25             2 2024-01-18 17:05:00+00:00         1.990893e+01 2024-01-18 17:05:00+00:00
18           emailservice       cpu  1.971771e+01            25             2 2024-01-18 17:15:00+00:00         1.976048e+01 2024-01-18 17:15:00+00:00
19              adservice  workload  1.550042e+01            25             2 2024-01-18 17:04:00+00:00         1.560000e+01 2024-01-18 17:04:00+00:00```
```

In [ ]:
# In[3]:

import pandas as pd

# Load trace.csv and parse timestamp as UTC datetime
trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, trace_name) using the full series
global_p95_trace = (
    trace
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) Merge global_p95 back into the trace series
trace = trace.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# 3) Filter trace to the incident window (UTC, inclusive)
window_start = pd.Timestamp("2024-01-18 16:58:05", tz='UTC')
window_end   = pd.Timestamp("2024-01-18 17:28:05", tz='UTC')
df_window = trace[(trace['timestamp'] >= window_start) & (trace['timestamp'] <= window_end)].copy()

# 4) For each (cmdb_id, trace_name) compute required summary fields
def summarize_group(g):
    count_window = len(g)
    exceed_mask = g['value'] > g['global_p95'].iloc[0]
    exceed_count = int(exceed_mask.sum())
    earliest_exceed_timestamp = g.loc[exceed_mask, 'timestamp'].min() if exceed_count > 0 else pd.NaT
    idxmax = g['value'].idxmax()
    max_value_in_window = g.loc[idxmax, 'value']
    max_value_timestamp = g.loc[idxmax, 'timestamp']
    return pd.Series({
        'cmdb_id': g['cmdb_id'].iloc[0],
        'trace_name': g['trace_name'].iloc[0],
        'global_p95': g['global_p95'].iloc[0],
        'count_window': count_window,
        'exceed_count': exceed_count,
        'earliest_exceed_timestamp': earliest_exceed_timestamp,
        'max_value_in_window': max_value_in_window,
        'max_value_timestamp': max_value_timestamp
    })

if not df_window.empty:
    trace_window_summary = (
        df_window
        .groupby(['cmdb_id', 'trace_name'], observed=True, sort=False)
        .apply(summarize_group)
        .reset_index(drop=True)
    )
    trace_window_summary = (
        trace_window_summary[trace_window_summary['count_window'] > 0]
        .sort_values(['exceed_count', 'max_value_in_window'], ascending=[False, False])
        .head(20)
        .reset_index(drop=True)
    )
else:
    trace_window_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','global_p95','count_window','exceed_count',
        'earliest_exceed_timestamp','max_value_in_window','max_value_timestamp'
    ])

# Return the compact summary
trace_window_summary

```
Out[3]:
```
Summary of trace analysis (incident window 2024-01-18 16:58:05 → 2024-01-18 17:28:05 UTC):

- The dominant anomalies are increased trace row_count values (request/row counts) across many services, with each affected trace showing 2 P95-exceeding points in the window.
- The primary hotspot is frontendservice (both incoming and outgoing traces):
  - trace.from_frontendservice.row_count and trace.to_frontendservice.row_count peaked at 17:05 UTC (max ~8334).
  - frontend → productcatalog and frontend → recommendation also show peaks at ~17:05 (and a secondary peak at ~17:10 for recommendation).
- Downstream services affected around the same time (~17:05):
  - productcatalogservice, recommendationservice, checkoutservice, currencyservice all show row_count exceedances at ~17:05 with their respective max values shortly after.
- There is a later/secondary spike for currencyservice row_count with max at 17:24 UTC.
- Pattern: a large request-volume surge begins around 17:05 UTC (originating at the frontend) propagating to productcatalog, recommendation, checkout, currency, etc., with a smaller later spike around 17:24 for currencyservice.

Conclusion / recommended next steps:
- This looks like a traffic/workload surge from the frontend around 17:05 UTC that propagated downstream. Investigate request sources, ingress/traffic logs, and rate-limiting or load-balancer activity around 17:05. Correlate with metrics (CPU, memory, disk I/O) and application logs at 17:05 and 17:24 to find any cascading resource pressure or failures.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                  trace_name  global_p95  count_window  exceed_count earliest_exceed_timestamp  max_value_in_window       max_value_timestamp
0         frontendservice        trace.from_frontendservice.row_count      8230.8            25             2 2024-01-18 17:05:00+00:00               8334.0 2024-01-18 17:05:00+00:00
1         frontendservice          trace.to_frontendservice.row_count      8230.8            25             2 2024-01-18 17:05:00+00:00               8334.0 2024-01-18 17:05:00+00:00
2         frontendservice    trace.to_productcatalogservice.row_count      3885.0            25             2 2024-01-18 17:05:00+00:00               3917.0 2024-01-18 17:05:00+00:00
3   productcatalogservice        trace.from_frontendservice.row_count      3885.0            25             2 2024-01-18 17:05:00+00:00               3917.0 2024-01-18 17:05:00+00:00
4         currencyservice        trace.from_frontendservice.row_count      2423.6            25             2 2024-01-18 17:18:00+00:00               2437.0 2024-01-18 17:24:00+00:00
5         frontendservice          trace.to_currencyservice.row_count      2423.6            25             2 2024-01-18 17:18:00+00:00               2437.0 2024-01-18 17:24:00+00:00
6         frontendservice                   trace.from_root.row_count       850.2            25             2 2024-01-18 17:05:00+00:00                868.0 2024-01-18 17:05:00+00:00
7                    root          trace.to_frontendservice.row_count       850.2            25             2 2024-01-18 17:05:00+00:00                868.0 2024-01-18 17:05:00+00:00
8         frontendservice    trace.to_recommendationservice.row_count       593.2            25             2 2024-01-18 17:05:00+00:00                599.0 2024-01-18 17:10:00+00:00
9   productcatalogservice  trace.from_recommendationservice.row_count       593.2            25             2 2024-01-18 17:05:00+00:00                599.0 2024-01-18 17:10:00+00:00
10  recommendationservice        trace.from_frontendservice.row_count       593.2            25             2 2024-01-18 17:05:00+00:00                599.0 2024-01-18 17:10:00+00:00
11  recommendationservice  trace.from_recommendationservice.row_count       593.2            25             2 2024-01-18 17:05:00+00:00                599.0 2024-01-18 17:10:00+00:00
12  recommendationservice    trace.to_productcatalogservice.row_count       593.2            25             2 2024-01-18 17:05:00+00:00                599.0 2024-01-18 17:10:00+00:00
13  recommendationservice    trace.to_recommendationservice.row_count       593.2            25             2 2024-01-18 17:05:00+00:00                599.0 2024-01-18 17:10:00+00:00
14        checkoutservice                   trace.from_root.row_count       146.6            25             2 2024-01-18 17:05:00+00:00                162.0 2024-01-18 17:05:00+00:00
15                   root          trace.to_checkoutservice.row_count       146.6            25             2 2024-01-18 17:05:00+00:00                162.0 2024-01-18 17:05:00+00:00
16        checkoutservice          trace.to_currencyservice.row_count       122.6            25             2 2024-01-18 17:05:00+00:00                138.0 2024-01-18 17:05:00+00:00
17        currencyservice        trace.from_checkoutservice.row_count       122.6            25             2 2024-01-18 17:05:00+00:00                138.0 2024-01-18 17:05:00+00:00
18        checkoutservice    trace.to_productcatalogservice.row_count        92.4            25             2 2024-01-18 17:05:00+00:00                100.0 2024-01-18 17:05:00+00:00
19  productcatalogservice        trace.from_checkoutservice.row_count        92.4            25             2 2024-01-18 17:05:00+00:00                100.0 2024-01-18 17:05:00+00:00```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Load metric.csv fresh and parse timestamp as UTC
metric = pd.read_csv("metric.csv")
metric['timestamp'] = pd.to_datetime(metric['timestamp'], unit='s', utc=True)

# Focus components
selected_components = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]

# Compute global P95 per (cmdb_id, kpi_name) across the full series for selected components
global_p95_df = (
    metric[metric['cmdb_id'].isin(selected_components)]
    .groupby(['cmdb_id', 'kpi_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# Build a lookup dict for global P95
gp95_lookup = {
    (row['cmdb_id'], row['kpi_name']): row['global_p95']
    for _, row in global_p95_df.iterrows()
}

# Incident window (UTC inclusive)
window_start = pd.Timestamp("2024-01-18 16:58:05", tz='UTC')
window_end   = pd.Timestamp("2024-01-18 17:28:05", tz='UTC')

# Filter metric to selected components and the incident window
df_window = metric[
    metric['cmdb_id'].isin(selected_components) &
    (metric['timestamp'] >= window_start) &
    (metric['timestamp'] <= window_end)
].copy()

# If no data in window, prepare empty DataFrame
if df_window.empty:
    metric_selected_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','count_window','exceed_count',
        'earliest_exceed_timestamp','max_value_in_window','max_value_timestamp','severity_ratio'
    ])
else:
    # Group and summarize, using gp95_lookup for global_p95 values
    def summarize_group(g):
        cmdb = g['cmdb_id'].iloc[0]
        kpi = g['kpi_name'].iloc[0]
        gp95 = gp95_lookup.get((cmdb, kpi), np.nan)
        count_window = len(g)
        exceed_mask = g['value'] > gp95 if not np.isnan(gp95) else np.array([False]*len(g))
        exceed_count = int(exceed_mask.sum())
        earliest_exceed_timestamp = g.loc[exceed_mask, 'timestamp'].min() if exceed_count > 0 else pd.NaT
        idxmax = g['value'].idxmax()
        max_value_in_window = float(g.loc[idxmax, 'value'])
        max_value_timestamp = g.loc[idxmax, 'timestamp']
        severity_ratio = (max_value_in_window / gp95) if (gp95 not in (0, 0.0, None, np.nan)) else np.nan
        return pd.Series({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'global_p95': gp95,
            'count_window': count_window,
            'exceed_count': exceed_count,
            'earliest_exceed_timestamp': earliest_exceed_timestamp,
            'max_value_in_window': max_value_in_window,
            'max_value_timestamp': max_value_timestamp,
            'severity_ratio': severity_ratio
        })

    metric_selected_summary = (
        df_window
        .groupby(['cmdb_id', 'kpi_name'], observed=True, sort=False)
        .apply(summarize_group)
        .reset_index(drop=True)
    )

    # Keep only rows with count_window > 0, sort by severity_ratio desc then exceed_count desc, limit to top 50
    metric_selected_summary = (
        metric_selected_summary[metric_selected_summary['count_window'] > 0]
        .sort_values(['severity_ratio', 'exceed_count'], ascending=[False, False])
        .head(50)
        .reset_index(drop=True)
    )

metric_selected_summary

```
Out[4]:
```
Summary (selected services, incident window 2024-01-18 16:58:05 → 2024-01-18 17:28:05 UTC):

Key anomalies (sorted by severity_ratio):
- checkoutservice — latency-50: global P95 = 0.3957, window count = 25, exceed_count = 2. Earliest exceed at 2024-01-18 17:17:00 UTC, max = 0.65195 at 17:17. Severity ratio ≈ 1.65 (largest).
- emailservice — latency-50: global P95 = 0.3119, window count = 25, exceed_count = 2. Earliest exceed at 17:17:00, max = 0.48562 at 17:17. Severity ratio ≈ 1.56.
- currencyservice — socket: global P95 ≈ 4.1067, window count = 25, exceed_count = 2. Earliest exceed at 17:23:00, max = 4.5 at 17:23. Severity ratio ≈ 1.10.
- productcatalogservice — workload: global P95 ≈ 147.50, window count = 25, exceed_count = 2. Earliest exceed at 17:05:00, max ≈ 154.21 at 17:05. Severity ratio ≈ 1.05.
- currencyservice — latency-90: global P95 ≈ 0.03876, window count = 25, exceed_count = 2. Earliest exceed at 17:05:00, max ≈ 0.04044 at 17:24. Severity ratio ≈ 1.04.

Additional notes:
- Almost all reported rows show count_window = 25 and exceed_count = 2 (i.e., two points in the window exceeded the global P95).
- Some KPI series (diskio, error) have global P95 = 0, producing NaN severity ratios; these had no exceedances in the window.
- Timing pattern: notable workload/row_count spikes around ~17:05 (productcatalog and related services) and latency spikes around ~17:17 (checkoutservice and emailservice). socket and some latency spikes appear later (~17:23–17:24).

Recommended next steps:
- Investigate the frontend/ingress traffic around ~17:05 (workload surge) and around ~17:17 (latency spikes) to find the triggering source.
- Correlate these timestamps with traces and logs for checkoutservice and productcatalogservice, and check resource metrics (CPU/memory/disk I/O) on the implicated services.
- Look at connection/socket behavior on currencyservice around 17:23–17:24.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  global_p95  count_window  exceed_count earliest_exceed_timestamp  max_value_in_window       max_value_timestamp  severity_ratio
0         checkoutservice  latency-50    0.395737            25             2 2024-01-18 17:17:00+00:00             0.651954 2024-01-18 17:17:00+00:00        1.647441
1            emailservice  latency-50    0.311868            25             2 2024-01-18 17:17:00+00:00             0.485619 2024-01-18 17:17:00+00:00        1.557129
2         currencyservice      socket    4.106667            25             2 2024-01-18 17:23:00+00:00             4.500000 2024-01-18 17:23:00+00:00        1.095779
3   productcatalogservice    workload  147.504833            25             2 2024-01-18 17:05:00+00:00           154.205567 2024-01-18 17:05:00+00:00        1.045427
4         currencyservice  latency-90    0.038757            25             2 2024-01-18 17:05:00+00:00             0.040435 2024-01-18 17:24:00+00:00        1.043302
..                    ...         ...         ...           ...           ...                       ...                  ...                       ...             ...
31        currencyservice      diskio    0.000000            25             0                       NaT             0.000000 2024-01-18 17:01:00+00:00             NaN
32           emailservice      diskio    0.000000            25             0                       NaT             0.000000 2024-01-18 17:01:00+00:00             NaN
33  productcatalogservice      diskio    0.000000            25             0                       NaT             0.000000 2024-01-18 17:01:00+00:00             NaN
34  productcatalogservice       error    0.000000            25             0                       NaT             0.000000 2024-01-18 17:01:00+00:00             NaN
35  recommendationservice      diskio    0.000000            25             0                       NaT             0.000000 2024-01-18 17:01:00+00:00             NaN

[36 rows x 9 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Load trace.csv and parse timestamp as UTC datetime (reuse variable name)
trace = pd.read_csv("trace.csv")
trace['timestamp'] = pd.to_datetime(trace['timestamp'], unit='s', utc=True)

# Components and window
selected_components = [
    "checkoutservice",
    "currencyservice",
    "emailservice",
    "productcatalogservice",
    "recommendationservice"
]
window_start = pd.Timestamp("2024-01-18 16:58:05", tz='UTC')
window_end   = pd.Timestamp("2024-01-18 17:28:05", tz='UTC')

# Compute global P95 per (cmdb_id, trace_name) using the full series
global_p95_trace = (
    trace
    .groupby(['cmdb_id', 'trace_name'], observed=True)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# Filter to selected components, trace_name containing 'duration', and the incident window
mask_component = trace['cmdb_id'].isin(selected_components)
mask_name = trace['trace_name'].str.contains('duration', na=False)
mask_window = (trace['timestamp'] >= window_start) & (trace['timestamp'] <= window_end)
df_window = trace[mask_component & mask_name & mask_window].copy()

# Merge global_p95 into the windowed data
df_window = df_window.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# Summarize per group
def summarize_group(g):
    gp95 = g['global_p95'].iloc[0]
    count_window = len(g)
    exceed_mask = g['value'] > gp95 if not pd.isna(gp95) else np.array([False]*len(g))
    exceed_count = int(exceed_mask.sum())
    earliest_exceed_timestamp = g.loc[exceed_mask, 'timestamp'].min() if exceed_count > 0 else pd.NaT
    idxmax = g['value'].idxmax()
    max_value_in_window = float(g.loc[idxmax, 'value'])
    max_value_timestamp = g.loc[idxmax, 'timestamp']
    severity_ratio = (max_value_in_window / gp95) if (gp95 not in (0, 0.0, None) and not pd.isna(gp95)) else np.nan
    return pd.Series({
        'cmdb_id': g['cmdb_id'].iloc[0],
        'trace_name': g['trace_name'].iloc[0],
        'global_p95': gp95,
        'count_window': count_window,
        'exceed_count': exceed_count,
        'earliest_exceed_timestamp': earliest_exceed_timestamp,
        'max_value_in_window': max_value_in_window,
        'max_value_timestamp': max_value_timestamp,
        'severity_ratio': severity_ratio
    })

if not df_window.empty:
    trace_duration_summary = (
        df_window
        .groupby(['cmdb_id', 'trace_name'], observed=True, sort=False)
        .apply(summarize_group)
        .reset_index(drop=True)
    )
    # Keep rows with count_window>0, sort by severity_ratio desc then exceed_count desc, limit to top 20
    trace_duration_summary = (
        trace_duration_summary[trace_duration_summary['count_window'] > 0]
        .sort_values(['severity_ratio', 'exceed_count'], ascending=[False, False], na_position='last')
        .head(20)
        .reset_index(drop=True)
    )
else:
    trace_duration_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','global_p95','count_window','exceed_count',
        'earliest_exceed_timestamp','max_value_in_window','max_value_timestamp','severity_ratio'
    ])

# Display compact summary
trace_duration_summary

```
Out[5]:
```
Summary (trace durations for selected services, window 2024-01-18 16:58:05 → 2024-01-18 17:28:05 UTC):

Key findings
- The largest relative duration spikes are for checkoutservice outbound calls:
  - checkoutservice → currencyservice (duration_p95): global P95 = 0.000275 s, max = 0.03748 s at 2024-01-18 17:25:00 UTC (severity_ratio ≈ 136). duration_mean also high (severity ≈ 9.68).
  - checkoutservice → emailservice: duration_p95 global P95 = 0.09123 s, max = 0.68740 s at 2024-01-18 17:20:00 UTC (severity_ratio ≈ 7.54). duration_mean also elevated.
- recommendationservice durations show substantial relative increases (duration_p95 and duration_mean), with maxes around 17:25 UTC (severity_ratios ≈ 6.6 and ~1.35 respectively).
- productcatalogservice and some root/from traces have notable duration_p95 increases (e.g., productcatalogservice trace.from_root.duration_p95 max ~0.0532 s at 17:24, severity ≈ 3.45).
- Most affected trace series have count_window = 25 and exceed_count = 2 (two points exceeded the global P95 in the window), with earliest exceedances appearing between ~17:09 and ~17:19 and the largest peaks at ~17:20–17:25.

Interpretation and recommended next steps
- The data points to checkoutservice outbound calls (especially to currencyservice and emailservice) as strong latency hotspots during the incident, with the most extreme relative spike seen at ~17:25 UTC for the checkout→currency path.
- Also investigate recommendationservice and productcatalogservice duration spikes around ~17:24–17:25 UTC.
- Actions: inspect traces and logs for checkoutservice outgoing requests around 17:09–17:25 UTC, check downstream service health (currencyservice, emailservice, recommendationservice, productcatalogservice), and correlate with resource metrics (CPU/memory/disk I/O) and ingress/workload spikes to determine root cause (network, resource saturation, or downstream slowness).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      trace_name  global_p95  count_window  exceed_count earliest_exceed_timestamp  max_value_in_window       max_value_timestamp  severity_ratio
0         checkoutservice           trace.to_currencyservice.duration_p95    0.000275            25             2 2024-01-18 17:09:00+00:00             0.037482 2024-01-18 17:25:00+00:00      136.141036
1         currencyservice         trace.from_checkoutservice.duration_p95    0.000275            25             2 2024-01-18 17:09:00+00:00             0.037482 2024-01-18 17:25:00+00:00      136.141036
2         checkoutservice          trace.to_currencyservice.duration_mean    0.002045            25             2 2024-01-18 17:09:00+00:00             0.019792 2024-01-18 17:25:00+00:00        9.675778
3         currencyservice        trace.from_checkoutservice.duration_mean    0.002045            25             2 2024-01-18 17:09:00+00:00             0.019792 2024-01-18 17:25:00+00:00        9.675778
4         checkoutservice              trace.to_emailservice.duration_p95    0.091227            25             2 2024-01-18 17:19:00+00:00             0.687401 2024-01-18 17:20:00+00:00        7.535076
5            emailservice         trace.from_checkoutservice.duration_p95    0.091227            25             2 2024-01-18 17:19:00+00:00             0.687401 2024-01-18 17:20:00+00:00        7.535076
6   recommendationservice   trace.from_recommendationservice.duration_p95    0.006470            25             2 2024-01-18 17:13:00+00:00             0.042700 2024-01-18 17:25:00+00:00        6.599364
7   recommendationservice     trace.to_recommendationservice.duration_p95    0.006470            25             2 2024-01-18 17:13:00+00:00             0.042700 2024-01-18 17:25:00+00:00        6.599364
8   recommendationservice         trace.from_frontendservice.duration_p95    0.007349            25             2 2024-01-18 17:17:00+00:00             0.043221 2024-01-18 17:25:00+00:00        5.881485
9   productcatalogservice                    trace.from_root.duration_p95    0.015440            25             2 2024-01-18 17:09:00+00:00             0.053235 2024-01-18 17:24:00+00:00        3.447844
10        checkoutservice            trace.to_paymentservice.duration_p95    0.001092            25             2 2024-01-18 17:13:00+00:00             0.003084 2024-01-18 17:13:00+00:00        2.824719
11        currencyservice                    trace.from_root.duration_p95    0.000128            25             2 2024-01-18 17:16:00+00:00             0.000238 2024-01-18 17:16:00+00:00        1.859401
12        checkoutservice             trace.to_emailservice.duration_mean    0.061880            25             2 2024-01-18 17:19:00+00:00             0.101048 2024-01-18 17:20:00+00:00        1.632980
13           emailservice        trace.from_checkoutservice.duration_mean    0.061880            25             2 2024-01-18 17:19:00+00:00             0.101048 2024-01-18 17:20:00+00:00        1.632980
14        checkoutservice           trace.to_paymentservice.duration_mean    0.000441            25             2 2024-01-18 17:04:00+00:00             0.000668 2024-01-18 17:13:00+00:00        1.515576
15  productcatalogservice  trace.from_recommendationservice.duration_mean    0.000021            25             2 2024-01-18 17:14:00+00:00             0.000029 2024-01-18 17:21:00+00:00        1.403652
16  recommendationservice    trace.to_productcatalogservice.duration_mean    0.000021            25             2 2024-01-18 17:14:00+00:00             0.000029 2024-01-18 17:21:00+00:00        1.403652
17  recommendationservice  trace.from_recommendationservice.duration_mean    0.005992            25             2 2024-01-18 17:21:00+00:00             0.008065 2024-01-18 17:25:00+00:00        1.346038
18  recommendationservice    trace.to_recommendationservice.duration_mean    0.005992            25             2 2024-01-18 17:21:00+00:00             0.008065 2024-01-18 17:25:00+00:00        1.346038
19  recommendationservice        trace.from_frontendservice.duration_mean    0.006550            25             2 2024-01-18 17:21:00+00:00             0.008615 2024-01-18 17:25:00+00:00        1.315187```
```